In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("training_runs")
SMOOTH = 80
SIGMA_ANCHOR = 0.85  # current config value

EXPERIMENTS = {
    "census": "SPECv3_census_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603092248_97f273d4",
    "credit": "SPECv3_credit_ablation_global_anchors_EP6000_PCA10_REWfairness_minID1_majID0_TRJ2000_REAL3000_GG202603071844_7fee08dc",
}
SEEDS = ["seed_42", "seed_123", "seed_999"]
SEED_COLORS = {"seed_42": "steelblue", "seed_123": "darkorange", "seed_999": "seagreen"}

def load_all(exp_folder):
    out = {}
    for s in SEEDS:
        p = ROOT / exp_folder / s / "metrics.csv"
        if p.is_file():
            out[s] = pd.read_csv(p)
    return out

def agg(seed_dfs, col, smooth=SMOOTH):
    eps = sorted(set().union(*[set(df["episode"].tolist()) for df in seed_dfs.values()]))
    arr = []
    for df in seed_dfs.values():
        if col in df.columns:
            s = df.set_index("episode")[col].reindex(eps)
            arr.append(s.values)
    if not arr:
        return np.array(eps), np.full(len(eps), np.nan), np.full(len(eps), np.nan)
    mat = np.array(arr, dtype=float)
    mu = np.nanmean(mat, axis=0)
    sd = np.nanstd(mat, axis=0)
    if smooth and smooth > 1:
        mu = pd.Series(mu).rolling(smooth, center=True, min_periods=1).mean().values
        sd = pd.Series(sd).rolling(smooth, center=True, min_periods=1).mean().values
    return np.array(eps), mu, sd

def get_transitions(seed_dfs):
    col = "align.curriculum_stage"
    for df in seed_dfs.values():
        if col in df.columns:
            t = df[df[col].diff().fillna(0) != 0]["episode"].tolist()
            return [x for x in t if x > 1]
    return []

def vlines(ax, transitions):
    for t in transitions:
        ax.axvline(t, color='gray', alpha=0.35, lw=1, ls='--')

data = {k: load_all(v) for k, v in EXPERIMENTS.items()}
print("Loaded:", {k: list(v.keys()) for k, v in data.items()})
print("Columns:", list(list(data['census'].values())[0].columns))

In [ ]:
# ============================================================
# CELL 2 — Summary table: early / mid / late means + trend
# ============================================================

ALL_METRICS = [
    ("global.global_obj",                 "global_obj"),
    ("global.local_reward",               "local_reward (combined)"),
    ("align.lambda_t",                    "lambda_t"),
    ("align.delta_global",                "delta_global"),
    ("align.corr_local_delta",            "corr_local_delta"),
    ("meta.avg_reward",                   "avg_reward"),
    ("meta.episode_return",               "episode_return"),
    ("utility.f1_minority_beta",          "F1_minority_beta"),
    ("utility.f1_macro_beta",             "F1_macro_beta"),
    ("utility.auc_beta",                  "ROC-AUC_beta"),
    ("utility.acc_beta",                  "acc_beta"),
    ("fairness.eo_tpr_diff",              "EO_tpr_diff  [EVAL]"),
    ("fairness.dp_diff",                  "DP_diff"),
    ("fairness.eod_max_diff",             "EOd_max_diff"),
    ("fairness.worst_loss_beta",          "worst_loss_beta"),
    ("fairness.worst_loss_alpha_baseline","worst_loss_alpha  [constant]"),
    ("fairness.group_loss_beta_g0",       "group_loss_g0 (maj)"),
    ("fairness.group_loss_beta_g1",       "group_loss_g1 (min)"),
    ("fairness.group_loss_gap_beta",      "group_loss_gap"),
    ("fairness.bce_mean_beta",            "bce_mean_beta"),
    ("local.anchor_reward_mean",          "anchor_reward  [sigma=0.85]"),
    ("local.min_anchor_dist_mean",        "min_anchor_dist"),
    ("local.hard_reward_mean",            "hard_reward"),
    ("local.div_pen_mean",                "div_pen"),
    ("local.local_clip_frac_0",           "clip_frac_floor"),
    ("local.local_clip_frac_1",           "clip_frac_ceil"),
    ("extra.diag_gen_radius_mean",        "gen_radius  [clip=3.0]"),
    ("extra.diag_mean_abs_margin",        "mean_abs_margin |p-0.5|"),
    ("extra.diag_frac_mid_conf",          "frac_mid_conf [0.4,0.6]"),
]

for dataset in ["census", "credit"]:
    seed_dfs = data[dataset]
    all_rows = pd.concat(seed_dfs.values(), ignore_index=True)
    early = all_rows[all_rows["episode"] <= 500]
    mid   = all_rows[(all_rows["episode"] >= 2000) & (all_rows["episode"] <= 4000)]
    late  = all_rows[all_rows["episode"] >= 5000]

    print(f"\n{'='*90}")
    print(f"  {dataset.upper()} — Early(ep1-500) / Mid(ep2k-4k) / Late(ep5k+)")
    print(f"{'='*90}")
    print(f"{'Metric':<42} {'Early':>10} {'Mid':>10} {'Late':>10} {'Δ(late-early)':>15} Trend")
    print("-"*92)
    for col, name in ALL_METRICS:
        if col not in all_rows.columns:
            continue
        e = early[col].mean()
        m = mid[col].mean()
        l = late[col].mean()
        if np.isnan(e) and np.isnan(l):
            continue
        delta = l - e if not (np.isnan(l) or np.isnan(e)) else np.nan
        if np.isnan(delta) or abs(delta) < 0.001:
            trend = "→"
        elif delta > 0:
            trend = "↑"
        else:
            trend = "↓"
        print(f"{name:<42} {e:>10.5f} {m:>10.5f} {l:>10.5f} {delta:>15.5f}   {trend}")

In [ ]:
# ============================================================
# CELL 3 — Critical diagnostic report (numbered findings)
# ============================================================

def diagnostic_report(dataset, seed_dfs):
    all_rows = pd.concat(seed_dfs.values(), ignore_index=True)
    early = all_rows[all_rows["episode"] <= 500]
    late  = all_rows[all_rows["episode"] >= 5000]

    print(f"\n{'='*72}")
    print(f"  DIAGNOSTIC REPORT — {dataset.upper()}")
    print(f"{'='*72}")

    # [1] Anchor reward vs sigma
    dist_e = early["local.min_anchor_dist_mean"].mean()
    dist_l = late["local.min_anchor_dist_mean"].mean()
    anc_e  = early["local.anchor_reward_mean"].mean()
    anc_l  = late["local.anchor_reward_mean"].mean()
    exp_e  = np.exp(-0.5 * (dist_e / SIGMA_ANCHOR)**2)
    exp_l  = np.exp(-0.5 * (dist_l / SIGMA_ANCHOR)**2)
    sigma_for_30pct = dist_l / np.sqrt(-2 * np.log(0.3))
    print(f"\n[1] ANCHOR REWARD vs SIGMA")
    print(f"    sigma_anchor = {SIGMA_ANCHOR}")
    print(f"    Min anchor dist:    early={dist_e:.3f}   late={dist_l:.3f}")
    print(f"    Expected reward:    early={exp_e:.6f}  late={exp_l:.8f}")
    print(f"    Actual reward:      early={anc_e:.6f}  late={anc_l:.6f}")
    print(f"    --> Sigma needed for reward>=0.30 at late dist: {sigma_for_30pct:.2f}")
    if anc_l < 0.005:
        print(f"    *** CRITICAL: Anchor reward is ZERO in late training. sigma={SIGMA_ANCHOR} is {dist_l/SIGMA_ANCHOR:.1f}x too small. ***")

    # [2] Global objective
    g_all  = all_rows["global.global_obj"]
    g_late = late["global.global_obj"]
    frac_above = (g_all > 0.55).mean()
    frac_below = (g_all < 0.45).mean()
    print(f"\n[2] GLOBAL OBJECTIVE (0.5 = neutral)")
    print(f"    Overall mean={g_all.mean():.4f}   late mean={g_late.mean():.4f}")
    print(f"    Frac > 0.55: {frac_above:.3f}   Frac < 0.45: {frac_below:.3f}")
    if g_all.mean() < 0.50:
        print(f"    *** CRITICAL: Beta is on average WORSE than alpha (mean < 0.5) ***")
    elif frac_above < 0.10:
        print(f"    *** WARNING: global_obj rarely exceeds 0.55 — extremely weak signal ***")

    # [3] Beta vs alpha worst loss
    alpha_base  = all_rows["fairness.worst_loss_alpha_baseline"].median()
    beta_late   = late["fairness.worst_loss_beta"].mean()
    frac_better = (all_rows["fairness.worst_loss_beta"] < all_rows["fairness.worst_loss_alpha_baseline"]).mean()
    print(f"\n[3] BETA vs ALPHA WORST-GROUP LOSS")
    print(f"    Alpha baseline (constant): {alpha_base:.5f}")
    print(f"    Beta worst loss (late):    {beta_late:.5f}")
    print(f"    Frac episodes beta < alpha: {frac_better:.3f}")
    if frac_better < 0.3:
        print(f"    *** WARNING: Beta beats alpha only {frac_better*100:.0f}% of the time ***")

    # [4] EO trend and per-seed variance
    eo_e = early["fairness.eo_tpr_diff"].mean()
    eo_l = late["fairness.eo_tpr_diff"].mean()
    eo_per = {s: df[df["episode"]>=5000]["fairness.eo_tpr_diff"].mean() for s,df in seed_dfs.items()}
    eo_vals = list(eo_per.values())
    eo_range = max(eo_vals) - min(eo_vals)
    print(f"\n[4] EO TPR DIFF (main eval metric, lower=better)")
    print(f"    Early mean: {eo_e:.4f}   Late mean: {eo_l:.4f}   Δ={eo_l-eo_e:+.4f}")
    print(f"    Per-seed late: " + "  ".join(f"{s}={v:.4f}" for s,v in eo_per.items()))
    print(f"    Range across seeds: {eo_range:.4f}  ({'HIGH variance' if eo_range > 0.05 else 'OK'})")

    # [5] Reward saturation
    c0 = all_rows["local.local_clip_frac_0"].mean()
    c1 = all_rows["local.local_clip_frac_1"].mean()
    print(f"\n[5] LOCAL REWARD SATURATION")
    print(f"    At floor (<=0): {c0:.4f}   At ceiling (>=1): {c1:.4f}")
    if c0 > 0.2:
        print(f"    *** WARNING: {c0*100:.0f}% of steps at reward=0 ***")
    if c1 > 0.2:
        print(f"    *** WARNING: {c1*100:.0f}% of steps saturated at 1 ***")

    # [6] Agent learning
    r_e = early["meta.avg_reward"].mean()
    r_l = late["meta.avg_reward"].mean()
    pct = (r_l - r_e) / (abs(r_e) + 1e-12) * 100
    print(f"\n[6] AGENT LEARNING (avg_reward per step)")
    print(f"    Early={r_e:.7f}  Late={r_l:.7f}  Change={pct:+.1f}%")
    if abs(pct) < 5:
        print(f"    *** WARNING: avg_reward barely changes — very weak learning signal ***")

    # [7] Generated radius
    rad_e = early["extra.diag_gen_radius_mean"].mean()
    rad_l = late["extra.diag_gen_radius_mean"].mean()
    print(f"\n[7] GENERATED POINT RADIUS (radius_clip=3.0)")
    print(f"    Early={rad_e:.3f}  Late={rad_l:.3f}")
    if rad_l > 2.7:
        print(f"    *** NOTE: Points at {rad_l:.2f} — near/hitting radius_clip boundary ***")

    # [8] Local-global correlation
    corr = all_rows["align.corr_local_delta"].dropna().mean()
    print(f"\n[8] LOCAL-GLOBAL CORRELATION")
    print(f"    Mean corr(local_reward, delta_global)={corr:.4f}")
    if corr < 0.05:
        print(f"    *** WARNING: Local reward has NO correlation with fairness improvement ***")

    # [9] Confidence of generated samples
    mg_e = early["extra.diag_mean_abs_margin"].mean()
    mg_l = late["extra.diag_mean_abs_margin"].mean()
    mf_l = late["extra.diag_frac_mid_conf"].mean()
    print(f"\n[9] CONFIDENCE OF GENERATED SAMPLES (alpha model)")
    print(f"    mean|p-0.5|: early={mg_e:.4f}  late={mg_l:.4f}  (0=uncertain, 0.5=confident)")
    print(f"    frac in [0.4,0.6]: late={mf_l:.4f}")
    if mg_l > 0.30:
        print(f"    *** NOTE: Generated samples are high-confidence predictions — NOT near boundary ***")

    # [10] Reward-fairness alignment
    print(f"\n[10] REWARD vs FAIRNESS ALIGNMENT")
    print(f"    The global reward optimizes: worst-group BCE loss across 4 intersection groups")
    print(f"    The eval metric is: EO TPR gap (threshold-based TPR difference)")
    # check if worst_loss improvement correlates with EO improvement episode-over-episode
    combined = pd.concat(seed_dfs.values(), ignore_index=True)
    # per-episode mean
    ep_means = combined.groupby("episode")[["fairness.worst_loss_beta", "fairness.eo_tpr_diff"]].mean()
    corr_wl_eo = ep_means["fairness.worst_loss_beta"].corr(ep_means["fairness.eo_tpr_diff"])
    print(f"    Pearson corr(worst_loss_beta, eo_tpr_diff) over episodes: {corr_wl_eo:.4f}")
    if abs(corr_wl_eo) < 0.3:
        print(f"    *** WARNING: Reward proxy (worst-group BCE) is weakly correlated with EO gap ***")
    elif corr_wl_eo > 0.3:
        print(f"    *** NOTE: Higher worst-loss is POSITIVELY correlated with worse EO — reward is well-aligned ***")

for ds, seed_dfs in data.items():
    diagnostic_report(ds, seed_dfs)

In [ ]:
# ============================================================
# Figure 1 — Global reward signals
# ============================================================
COLS_GLOBAL = [
    ("global.global_obj",   "Global Obj (0.5=neutral, red line)"),
    ("global.local_reward", "Local Reward (combined)"),
    ("align.lambda_t",      "Lambda_t (global weight schedule)"),
    ("align.delta_global",  "Delta Global (episode-over-episode)"),
]

fig, axes = plt.subplots(len(COLS_GLOBAL), 2, figsize=(16, 3.8*len(COLS_GLOBAL)), constrained_layout=True)
fig.suptitle("Figure 1: Global Reward Signals", fontsize=14, fontweight="bold")
for row, (col, title) in enumerate(COLS_GLOBAL):
    for cidx, ds in enumerate(["census", "credit"]):
        ax = axes[row, cidx]
        eps, mu, sd = agg(data[ds], col)
        ax.plot(eps, mu, lw=2, color='steelblue')
        ax.fill_between(eps, mu-sd, mu+sd, alpha=0.25, color='steelblue')
        if col == "global.global_obj":
            ax.axhline(0.5, color='red', alpha=0.7, lw=1.2, ls=':')
            ax.set_ylim(0.3, 0.65)
        vlines(ax, get_transitions(data[ds]))
        ax.set_title(f"{ds.upper()} — {title}", fontsize=9)
        ax.set_xlabel("Episode")
        ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Figure 2 — Fairness metrics during training (with per-seed lines)
# ============================================================
COLS_FAIR = [
    ("fairness.eo_tpr_diff",  "EO |ΔTPR|  [MAIN EVAL, ↓ better]"),
    ("fairness.dp_diff",      "DP gap"),
    ("fairness.eod_max_diff", "EOd max"),
]
fig, axes = plt.subplots(len(COLS_FAIR), 2, figsize=(16, 4*len(COLS_FAIR)), constrained_layout=True)
fig.suptitle("Figure 2: Fairness Metrics (dashed = individual seeds)", fontsize=14, fontweight="bold")
for row, (col, title) in enumerate(COLS_FAIR):
    for cidx, ds in enumerate(["census", "credit"]):
        ax = axes[row, cidx]
        eps, mu, sd = agg(data[ds], col)
        ax.plot(eps, mu, lw=2.5, color='crimson', label='mean', zorder=3)
        ax.fill_between(eps, mu-sd, mu+sd, alpha=0.2, color='crimson', zorder=2)
        for s, df in data[ds].items():
            if col in df.columns:
                y = df[col].rolling(SMOOTH, center=True, min_periods=1).mean()
                ax.plot(df["episode"], y, lw=1, alpha=0.6, ls='--', color=SEED_COLORS[s], label=s)
        vlines(ax, get_transitions(data[ds]))
        ax.set_title(f"{ds.upper()} — {title}", fontsize=9)
        ax.set_xlabel("Episode")
        ax.legend(fontsize=8)
        ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Figure 3 — DRO worst-group loss: Beta vs Alpha + per-group breakdown
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
fig.suptitle("Figure 3: DRO Worst-Group Loss", fontsize=14, fontweight="bold")
for cidx, ds in enumerate(["census", "credit"]):
    sd = data[ds]
    tr = get_transitions(sd)
    # top: beta vs alpha
    ax = axes[0, cidx]
    eps, mu_b, sd_b = agg(sd, "fairness.worst_loss_beta")
    eps, mu_a, _    = agg(sd, "fairness.worst_loss_alpha_baseline")
    ax.plot(eps, mu_b, lw=2, color='steelblue', label='beta worst-loss')
    ax.fill_between(eps, mu_b-sd_b, mu_b+sd_b, alpha=0.25, color='steelblue')
    ax.plot(eps, mu_a, lw=1.5, color='tomato', ls='--', label='alpha baseline')
    vlines(ax, tr)
    ax.set_title(f"{ds.upper()} — Worst-Group Loss (Beta vs Alpha)")
    ax.set_xlabel("Episode")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    # bottom: per-group breakdown
    ax2 = axes[1, cidx]
    eps, mu_g0, sd_g0 = agg(sd, "fairness.group_loss_beta_g0")
    eps, mu_g1, sd_g1 = agg(sd, "fairness.group_loss_beta_g1")
    ax2.plot(eps, mu_g0, lw=2, color='royalblue', label='g0 loss')
    ax2.plot(eps, mu_g1, lw=2, color='darkorange', label='g1 loss')
    ax2.fill_between(eps, mu_g0-sd_g0, mu_g0+sd_g0, alpha=0.2, color='royalblue')
    ax2.fill_between(eps, mu_g1-sd_g1, mu_g1+sd_g1, alpha=0.2, color='darkorange')
    vlines(ax2, tr)
    ax2.set_title(f"{ds.upper()} — Per-Group BCE Loss")
    ax2.set_xlabel("Episode")
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Figure 4 — Local reward breakdown
# ============================================================
COLS_LOCAL = [
    ("local.anchor_reward_mean",   "Anchor Reward (target > 0.10)"),
    ("local.min_anchor_dist_mean", f"Min Anchor Distance (sigma={SIGMA_ANCHOR} shown)"),
    ("local.hard_reward_mean",     "Hard Reward"),
    ("local.div_pen_mean",         "Diversity Penalty"),
]
fig, axes = plt.subplots(len(COLS_LOCAL), 2, figsize=(16, 4*len(COLS_LOCAL)), constrained_layout=True)
fig.suptitle("Figure 4: Local Reward Breakdown", fontsize=14, fontweight="bold")
for row, (col, title) in enumerate(COLS_LOCAL):
    for cidx, ds in enumerate(["census", "credit"]):
        ax = axes[row, cidx]
        eps, mu, sd = agg(data[ds], col)
        ax.plot(eps, mu, lw=2, color='darkorange')
        ax.fill_between(eps, mu-sd, mu+sd, alpha=0.25, color='darkorange')
        if col == "local.anchor_reward_mean":
            ax.axhline(0.10, color='red', alpha=0.6, lw=1.2, ls=':', label='floor=0.10')
            ax.legend(fontsize=8)
        if col == "local.min_anchor_dist_mean":
            ax.axhline(SIGMA_ANCHOR, color='red', alpha=0.7, lw=1.2, ls=':', label=f'sigma={SIGMA_ANCHOR}')
            ax.legend(fontsize=8)
        vlines(ax, get_transitions(data[ds]))
        ax.set_title(f"{ds.upper()} — {title}", fontsize=9)
        ax.set_xlabel("Episode")
        ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Figure 5 — Generated sample properties + saturation
# ============================================================
COLS_EXTRA = [
    ("extra.diag_gen_radius_mean", "Generated L2 Radius (radius_clip=3.0)"),
    ("extra.diag_mean_abs_margin", "Mean |p-0.5|  (0=uncertain, 0.5=confident)"),
    ("extra.diag_frac_mid_conf",   "Frac samples p in [0.4, 0.6]"),
    ("local.local_clip_frac_0",    "Local reward at floor (>=0)"),
    ("local.local_clip_frac_1",    "Local reward at ceiling (<=1)"),
]
fig, axes = plt.subplots(len(COLS_EXTRA), 2, figsize=(16, 4*len(COLS_EXTRA)), constrained_layout=True)
fig.suptitle("Figure 5: Generated Sample Properties & Saturation", fontsize=14, fontweight="bold")
for row, (col, title) in enumerate(COLS_EXTRA):
    for cidx, ds in enumerate(["census", "credit"]):
        ax = axes[row, cidx]
        eps, mu, sd = agg(data[ds], col)
        ax.plot(eps, mu, lw=2, color='purple')
        ax.fill_between(eps, mu-sd, mu+sd, alpha=0.25, color='purple')
        if col == "extra.diag_gen_radius_mean":
            ax.axhline(3.0, color='red', alpha=0.6, lw=1.2, ls=':', label='radius_clip=3.0')
            ax.legend(fontsize=8)
        vlines(ax, get_transitions(data[ds]))
        ax.set_title(f"{ds.upper()} — {title}", fontsize=9)
        ax.set_xlabel("Episode")
        ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Figure 6 — Meta / Align signals (reward + curriculum)
# ============================================================
COLS_META = [
    ("meta.avg_reward",         "Avg Reward per Step (agent learning?)"),
    ("meta.episode_return",     "Episode Return (cumulative)"),
    ("align.corr_local_delta",  "Corr(local_reward, delta_global)"),
    ("align.curriculum_stage",  "Curriculum Stage"),
]
fig, axes = plt.subplots(len(COLS_META), 2, figsize=(16, 4*len(COLS_META)), constrained_layout=True)
fig.suptitle("Figure 6: Meta & Alignment Signals", fontsize=14, fontweight="bold")
for row, (col, title) in enumerate(COLS_META):
    for cidx, ds in enumerate(["census", "credit"]):
        ax = axes[row, cidx]
        eps, mu, sd = agg(data[ds], col)
        ax.plot(eps, mu, lw=2, color='teal')
        ax.fill_between(eps, mu-sd, mu+sd, alpha=0.25, color='teal')
        if col == "align.corr_local_delta":
            ax.axhline(0, color='gray', alpha=0.6, lw=1, ls=':')
        vlines(ax, get_transitions(data[ds]))
        ax.set_title(f"{ds.upper()} — {title}", fontsize=9)
        ax.set_xlabel("Episode")
        ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Figure 7 — Sigma sweep: what sigma makes the anchor reward useful?
# ============================================================
SIGMA_SWEEP = [0.50, 0.85, 1.5, 2.5, 3.0, 4.0, 5.0]
COLORS_SWEEP = plt.cm.viridis(np.linspace(0.1, 0.9, len(SIGMA_SWEEP)))

fig, axes = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
fig.suptitle("Figure 7: Anchor Reward for Different Sigma Values (at observed distances)", fontsize=13, fontweight="bold")

for cidx, ds in enumerate(["census", "credit"]):
    ax = axes[cidx]
    eps, dist_mu, _ = agg(data[ds], "local.min_anchor_dist_mean")
    for i, sigma in enumerate(SIGMA_SWEEP):
        reward_curve = np.exp(-0.5 * (dist_mu / sigma)**2)
        lw  = 3.0 if sigma == SIGMA_ANCHOR else 1.5
        ls  = '--' if sigma == SIGMA_ANCHOR else '-'
        lbl = f"sigma={sigma}" + (" ← CURRENT" if sigma == SIGMA_ANCHOR else "")
        ax.plot(eps, reward_curve, lw=lw, ls=ls, color=COLORS_SWEEP[i], label=lbl)
    ax.axhline(0.10, color='red',    alpha=0.5, lw=1.2, ls=':', label='reward=0.10 (floor for usefulness)')
    ax.axhline(0.30, color='orange', alpha=0.5, lw=1.2, ls=':', label='reward=0.30 (target)')
    vlines(ax, get_transitions(data[ds]))
    ax.set_title(f"{ds.upper()}", fontsize=10)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Anchor Reward")
    ax.set_ylim(-0.05, 1.05)
    ax.legend(fontsize=8, loc='upper right')
    ax.grid(True, alpha=0.3)
plt.show()

print("\nAnchor reward at LATE (ep>=5000) mean min-anchor-distance:")
print(f"{'Dataset':<10}  {'Late dist':>10}", end="")
for s in SIGMA_SWEEP:
    marker = "(CUR)" if s == SIGMA_ANCHOR else ""
    print(f"  sig={s:.1f}{marker}", end="")
print()
for ds in ["census", "credit"]:
    rows = pd.concat(data[ds].values())
    d = rows[rows["episode"] >= 5000]["local.min_anchor_dist_mean"].mean()
    print(f"{ds:<10}  {d:>10.3f}", end="")
    for s in SIGMA_SWEEP:
        r = np.exp(-0.5 * (d / s)**2)
        print(f"  {r:>9.5f}", end="")
    print()

In [ ]:
# ============================================================
# Figure 8 — Per-seed EO and worst-group loss
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 9), constrained_layout=True)
fig.suptitle("Figure 8: Per-Seed Variance", fontsize=14, fontweight="bold")

METRICS_PERSEED = [
    ("fairness.eo_tpr_diff",     "EO |ΔTPR| (↓ better)"),
    ("fairness.worst_loss_beta", "Worst-Group Loss Beta (dashed=alpha baseline)"),
]
for row, (col, title) in enumerate(METRICS_PERSEED):
    for cidx, ds in enumerate(["census", "credit"]):
        ax = axes[row, cidx]
        sd = data[ds]
        for s, df in sd.items():
            if col in df.columns:
                y = df[col].rolling(SMOOTH, center=True, min_periods=1).mean()
                ax.plot(df["episode"], y, lw=2, color=SEED_COLORS[s], label=s)
        if col == "fairness.worst_loss_beta":
            for s, df in sd.items():
                if "fairness.worst_loss_alpha_baseline" in df.columns:
                    v = df["fairness.worst_loss_alpha_baseline"].iloc[0]
                    ax.axhline(v, color=SEED_COLORS[s], alpha=0.5, lw=1.5, ls='--')
        vlines(ax, get_transitions(sd))
        ax.set_title(f"{ds.upper()} — {title}", fontsize=9)
        ax.set_xlabel("Episode")
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)
plt.show()

print("\nPer-seed late (ep>=5000) summary:")
for ds in ["census", "credit"]:
    print(f"\n{ds.upper()}:")
    for s, df in data[ds].items():
        late = df[df["episode"] >= 5000]
        eo  = late["fairness.eo_tpr_diff"].mean()
        wl  = late["fairness.worst_loss_beta"].mean()
        al  = df["fairness.worst_loss_alpha_baseline"].iloc[0]
        tag = "BETTER" if wl < al else "WORSE "
        ar  = late["local.anchor_reward_mean"].mean()
        dist = late["local.min_anchor_dist_mean"].mean()
        print(f"  {s}: EO={eo:.4f}  worst_loss={wl:.5f} vs alpha={al:.5f} [{tag}]  anchor_rwd={ar:.6f}  dist={dist:.3f}")

In [ ]:
# ============================================================
# Figure 9 — Utility during training
# ============================================================
COLS_UTIL = [
    ("utility.f1_minority_beta", "F1 Minority"),
    ("utility.f1_macro_beta",    "F1 Macro"),
    ("utility.auc_beta",         "ROC-AUC"),
    ("utility.acc_beta",         "Accuracy"),
]
fig, axes = plt.subplots(len(COLS_UTIL), 2, figsize=(16, 4*len(COLS_UTIL)), constrained_layout=True)
fig.suptitle("Figure 9: Utility During Training", fontsize=14, fontweight="bold")
for row, (col, title) in enumerate(COLS_UTIL):
    for cidx, ds in enumerate(["census", "credit"]):
        ax = axes[row, cidx]
        eps, mu, sd = agg(data[ds], col)
        ax.plot(eps, mu, lw=2, color='forestgreen')
        ax.fill_between(eps, mu-sd, mu+sd, alpha=0.25, color='forestgreen')
        vlines(ax, get_transitions(data[ds]))
        ax.set_title(f"{ds.upper()} — {title}", fontsize=9)
        ax.set_xlabel("Episode")
        ax.grid(True, alpha=0.3)
plt.show()